# Granite Rationale Training (from JSONL)

Simple notebook to train Granite 3.2-2B on pre-generated rationale JSONL.

**Setup on Kaggle:**
1. Add dataset `gigibot/rationale-semeval2026` as Input
2. Enable GPU T4 x2
3. Run all cells

In [1]:
!pip -q install -U transformers datasets accelerate bitsandbytes peft pandas scikit-learn trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 13.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 110.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 106.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 36.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.1 requires pand

In [2]:
### Cell 1: Setup & Config
import os
import json
import random
from pathlib import Path
from collections import Counter
import glob

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

# Environment detection
IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB = 'COLAB_GPU' in os.environ
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment: Kaggle={IN_KAGGLE}, Colab={IN_COLAB}, Device={DEVICE}')

# ============ CONFIG ============
# Input JSONL - auto-detect from Kaggle input
def find_training_jsonl():
    """Find the RationaleTraining JSONL in Kaggle input directories."""
    candidates = [
        '/kaggle/input/rationale-semeval2026/RationaleTraining_raw.jsonl',
        '/kaggle/input/semeval/RationaleTraining_raw.jsonl',
        '/kaggle/input/datasets/gigibot/rationale-semeval2026/RationaleTraining_raw.jsonl'
    ]
    # Also search all input directories
    if os.path.exists('/kaggle/input'):
        for pattern in ['/kaggle/input/**/RationaleTraining*.jsonl', '/kaggle/input/**/*.jsonl']:
            candidates.extend(glob.glob(pattern, recursive=True))
    
    for c in candidates:
        if os.path.exists(c):
            return Path(c)
    return None

TRAINING_JSONL = find_training_jsonl()
if TRAINING_JSONL:
    print(f'✅ Found JSONL: {TRAINING_JSONL}')
else:
    # List what's available
    print('❌ JSONL not found! Available inputs:')
    if os.path.exists('/kaggle/input'):
        for d in os.listdir('/kaggle/input'):
            print(f'  /kaggle/input/{d}/')
            inp_dir = Path(f'/kaggle/input/{d}')
            for f in inp_dir.glob('*'):
                print(f'    - {f.name}')
    raise FileNotFoundError('Add dataset gigibot/rationale-semeval2026 as Input!')

# Model
BASE_MODEL = 'ibm-granite/granite-3.2-8b-instruct'
LOAD_4BIT = True  # 8B needs 4-bit quantization to fit on T4 GPUs

# Training
EPOCHS = 4  # Increased from 2 - loss was still dropping
BATCH_SIZE = 1
GRADIENT_ACCUM = 8
LEARNING_RATE = 2e-4
MAX_LENGTH = 1024
VALIDATION_SPLIT = 0.1
BALANCE_MODE = 'upsample'  # 'upsample', 'downsample', or None

# Training mode: 'sft' (standard) or 'grpo' (reasoning optimization)
TRAINING_MODE = 'sft'  # Start with SFT, switch to GRPO after base training

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Output
OUTPUT_DIR = Path('/kaggle/working/granite_lora_trained')

# Clean up any old incompatible checkpoints from previous runs (2B vs 8B mismatch)
import shutil
if OUTPUT_DIR.exists():
    print(f'⚠️ Removing old output directory to avoid checkpoint conflicts...')
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resume options (set to None for fresh training)
# Option 1: Load adapter weights only (restarts training from step 0)
# NOTE: Adapter must match base model! 2B adapter won't work with 8B model
RESUME_FROM_ADAPTER = None  # e.g., 'gigibot/granite-clarity-lora-8b' or '/kaggle/input/my-adapter'

# Option 2: Full checkpoint resume (continues from exact step, restores optimizer/scheduler)
# To use: upload checkpoint folder as Kaggle dataset, then set path here
RESUME_FROM_CHECKPOINT = None  # e.g., '/kaggle/input/my-checkpoint/checkpoint-32'

print(f'Training JSONL: {TRAINING_JSONL}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Resume adapter: {RESUME_FROM_ADAPTER or "None"}')
print(f'Resume checkpoint: {RESUME_FROM_CHECKPOINT or "None (fresh training)"}')

Environment: Kaggle=True, Colab=False, Device=cuda
✅ Found JSONL: /kaggle/input/datasets/gigibot/rationale-semeval2026/RationaleTraining_raw.jsonl
⚠️ Removing old output directory to avoid checkpoint conflicts...
Training JSONL: /kaggle/input/datasets/gigibot/rationale-semeval2026/RationaleTraining_raw.jsonl
Output dir: /kaggle/working/granite_lora_trained
Resume adapter: None
Resume checkpoint: None (fresh training)


In [3]:
### Cell 2: Load and Parse JSONL
import re

LABEL_MAP = {
    'Clear Reply': 'Direct Reply',
    'Clear Non-Reply': 'Direct Non-Reply',
    'Ambivalent': 'Indirect',
    'Ambivalent Reply': 'Indirect',
}

def map_label(raw):
    v = str(raw).strip()
    return LABEL_MAP.get(v, v if v in ['Direct Reply', 'Direct Non-Reply', 'Indirect'] else 'Indirect')

def parse_jsonl_record(rec):
    """Parse a JSONL record into Q, A, reasoning, label."""
    inp = rec.get('input', '')
    out = rec.get('output', '')
    
    # Parse Q/A from input
    m = re.match(r'Q:\s*(.*?)\nA:\s*(.*)', inp, re.DOTALL)
    if m:
        q, a = m.group(1).strip(), m.group(2).strip()
    else:
        q, a = '', inp
    
    # Parse reasoning and verdict from output
    reasoning = ''
    verdict = ''
    think = re.search(r'<think>(.*?)</think>', out, re.DOTALL | re.IGNORECASE)
    if think:
        reasoning = think.group(1).strip()
    vm = re.search(r'Verdict:\s*([^\n]+)', out, re.IGNORECASE)
    if vm:
        verdict = vm.group(1).strip()
    
    return {
        'question': q,
        'answer': a,
        'reasoning': reasoning,
        'label': map_label(verdict),
    }

# Load JSONL (already validated in setup cell)

rows = []
with open(TRAINING_JSONL, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        parsed = parse_jsonl_record(rec)
        if parsed['question'] and parsed['answer'] and parsed['label']:
            rows.append(parsed)

print(f'Loaded {len(rows)} examples')
label_counts = Counter(r['label'] for r in rows)
print(f'Label distribution: {dict(label_counts)}')

Loaded 123 examples
Label distribution: {'Indirect': 39, 'Direct Reply': 46, 'Direct Non-Reply': 38}


In [4]:
### Cell 3: Balance and Split Data

def upsample_to_max(rows):
    """Upsample minority classes to match majority."""
    by_label = {}
    for r in rows:
        by_label.setdefault(r['label'], []).append(r)
    max_count = max(len(v) for v in by_label.values())
    balanced = []
    for label, items in by_label.items():
        if len(items) < max_count:
            upsampled = items * (max_count // len(items)) + random.sample(items, max_count % len(items))
            balanced.extend(upsampled)
        else:
            balanced.extend(items)
    random.shuffle(balanced)
    return balanced

def downsample_to_min(rows):
    """Downsample majority classes to match minority."""
    by_label = {}
    for r in rows:
        by_label.setdefault(r['label'], []).append(r)
    min_count = min(len(v) for v in by_label.values())
    balanced = []
    for label, items in by_label.items():
        balanced.extend(random.sample(items, min_count))
    random.shuffle(balanced)
    return balanced

# Balance
if BALANCE_MODE == 'upsample':
    rows = upsample_to_max(rows)
    print(f'After upsampling: {len(rows)} examples')
elif BALANCE_MODE == 'downsample':
    rows = downsample_to_min(rows)
    print(f'After downsampling: {len(rows)} examples')

# Split
random.shuffle(rows)
n_val = int(len(rows) * VALIDATION_SPLIT)
val_rows = rows[:n_val]
train_rows = rows[n_val:]
print(f'Train: {len(train_rows)}, Validation: {len(val_rows)}')

After upsampling: 138 examples
Train: 125, Validation: 13


In [5]:
### Cell 4: Load Model and Tokenizer

print(f'Loading model: {BASE_MODEL}')
print(f'Quantization: {"4-bit" if LOAD_4BIT else "None (bf16)"}')

# Check available GPUs
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
total_vram = 0
print(f'Available GPUs: {N_GPUS}')
for i in range(N_GPUS):
    vram_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
    total_vram += vram_gb
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gb:.1f} GB)')
print(f'Total VRAM: {total_vram:.1f} GB')

# 8B model sizes:
# - bf16 (no quant): ~16GB weights + ~8-10GB activations = needs ~26GB (2x T4)
# - 4-bit: ~4GB weights + activations = fits on 1x T4
print(f'8B model estimate: {"~5GB (4-bit)" if LOAD_4BIT else "~16GB (bf16) - will use both GPUs"}')

# Quantization config
quant_config = None
if LOAD_4BIT and torch.cuda.is_available():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model - device_map='auto' distributes across available GPUs
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
print(f'✅ Model loaded on: {model.hf_device_map if hasattr(model, "hf_device_map") else "single device"}')

# LoRA - either load existing adapter or create new
from peft import PeftModel

if RESUME_FROM_ADAPTER:
    print(f'📥 Loading existing adapter from: {RESUME_FROM_ADAPTER}')
    model = PeftModel.from_pretrained(model, RESUME_FROM_ADAPTER)
    model.print_trainable_parameters()
    print('✅ Loaded existing LoRA adapter - will continue training')
else:
    print('🆕 Creating fresh LoRA adapter')
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

# Enable gradient checkpointing for memory efficiency
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

print(f'✅ Model ready (quantization: {"4-bit" if LOAD_4BIT else "bf16"})')

Loading model: ibm-granite/granite-3.2-8b-instruct
Quantization: 4-bit
Available GPUs: 2
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)
Total VRAM: 31.3 GB
8B model estimate: ~5GB (4-bit)


config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Model loaded on: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.layers.36': 1, 'model.layers.37': 1, 'model.layers.38': 1, 'model.layers.39': 1, 'model.norm': 1, 'model.rotary_emb': 1}
🆕 Creating fresh LoRA adapter
t

In [6]:
### Cell 5: Prepare Dataset

def build_prompt(q, a):
    return f"""You are analyzing political interview answers for clarity classification.

Question: {q}
Answer: {a}

Analyze the answer step-by-step:
1. Does it directly address the question?
2. Is it evasive or indirect?
3. Does it decline to answer?

Provide your reasoning and then classify as one of:
- "Direct Reply": Directly answers the question
- "Direct Non-Reply": Explicitly declines or claims inability to answer  
- "Indirect": Evasive, indirect, or partially answers

Respond in JSON format:
{{
  "reasoning": "Your step-by-step analysis...",
  "label": "Direct Reply|Direct Non-Reply|Indirect"
}}"""

def build_assistant(reasoning, label):
    return json.dumps({'reasoning': reasoning, 'label': label}, ensure_ascii=False)

def tokenize_example(row):
    user_msg = build_prompt(row['question'], row['answer'])
    assistant_msg = build_assistant(row['reasoning'], row['label'])
    
    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except:
        text = f"User: {user_msg}\n\nAssistant: {assistant_msg}"
    
    # Tokenize
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
        return_tensors=None,
    )
    
    # Labels = input_ids (for causal LM)
    encoded['labels'] = encoded['input_ids'].copy()
    
    return encoded

# Create datasets
train_data = [tokenize_example(r) for r in train_rows]
val_data = [tokenize_example(r) for r in val_rows]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f'Train dataset: {len(train_dataset)} examples')
print(f'Val dataset: {len(val_dataset)} examples')

Train dataset: 125 examples
Val dataset: 13 examples


In [21]:
### Cell 6b: Evaluate on SemEval Gold Labels (Task1)

# Runs evaluation on the official eval set + gold labels.
# Uses the same prompt/prediction style as training, with optional self-consistency voting.

RUN_GOLD_EVAL = True
GOLD_BATCH_SIZE = 3          # target parallel batch size (auto-fallback to 1 on OOM)
GOLD_NUM_SAMPLES = 3         # self-consistency votes per sample (computed in parallel)
GOLD_MAX_NEW_TOKENS = 300    # tracked below; warning printed if frequently capped
GOLD_TEMPERATURE = 0.6       # >0 required for diverse votes; use 0.0 for deterministic eval

if RUN_GOLD_EVAL:
    import csv
    import time

    # Resolve files from Kaggle input first, then local clone
    label_candidates = [
        '/kaggle/input/datasets/gigibot/claritysemevalevaldataset/task1_eval_labels.txt',
        '/kaggle/input/claritysemevalevaldataset/task1_eval_labels.txt',
        '/kaggle/input/datasets/gigibot/evalsetsemevalpolitical/task1_eval_labels.txt',
        '/kaggle/input/evalsetsemevalpolitical/task1_eval_labels.txt',
        '/Users/andrearachetta/Downloads/task1_eval_labels.txt',
    ]
    eval_csv_candidates = [
        '/kaggle/input/datasets/gigibot/claritysemevalevaldataset/clarity_task_evaluation_dataset.csv',
        '/kaggle/input/claritysemevalevaldataset/clarity_task_evaluation_dataset.csv',
        '/kaggle/input/datasets/gigibot/evalsetsemevalpolitical/clarity_task_evaluation_dataset.csv',
        '/kaggle/input/evalsetsemevalpolitical/clarity_task_evaluation_dataset.csv',
        '/kaggle/working/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv',
        'dataset/clarity_task_evaluation_dataset.csv',
    ]

    def first_existing(paths):
        for p in paths:
            if os.path.exists(p):
                return Path(p)
        return None

    labels_path = first_existing(label_candidates)
    eval_csv_path = first_existing(eval_csv_candidates)

    if labels_path is None:
        raise FileNotFoundError(f'Gold labels file not found. Checked: {label_candidates}')
    if eval_csv_path is None:
        raise FileNotFoundError(f'Eval CSV file not found. Checked: {eval_csv_candidates}')

    print(f'✅ Gold labels: {labels_path}')
    print(f'✅ Eval CSV:    {eval_csv_path}')

    with labels_path.open('r', encoding='utf-8') as f:
        raw_gold = [line.strip() for line in f if line.strip()]
    gold_labels = [map_label(x) for x in raw_gold]

    with eval_csv_path.open('r', encoding='utf-8', newline='') as f:
        eval_rows = list(csv.DictReader(f))

    # Build evaluation examples in the official order
    examples = []
    for r in eval_rows:
        q = str(r.get('question') or r.get('interview_question') or '').strip()
        a = str(r.get('interview_answer') or r.get('answer') or '').strip()
        idx_raw = str(r.get('index') or '').strip()
        row_idx = int(idx_raw) if idx_raw.isdigit() else len(examples)
        examples.append({'index': row_idx, 'question': q, 'answer': a})

    if len(examples) != len(gold_labels):
        raise ValueError(
            f'Row mismatch: eval_rows={len(examples)} vs gold_labels={len(gold_labels)}. '
            'Check dataset version/order.'
        )

    # Sort by index to guarantee alignment
    examples = sorted(examples, key=lambda x: x['index'])

    print(f'📏 Alignment OK: {len(examples)} samples (index {examples[0]["index"]}..{examples[-1]["index"]})')
    print('Gold label distribution:', dict(Counter(gold_labels)))

    model.eval()
    allowed = ['Direct Reply', 'Direct Non-Reply', 'Indirect']

    def parse_label(response_text):
        try:
            data = json.loads(response_text)
            lbl = data.get('label', 'Indirect')
            if lbl in allowed:
                return lbl
        except Exception:
            pass

        low = response_text.lower()
        for lbl in allowed:
            if lbl.lower() in low:
                return lbl
        return 'Indirect'

    def generate_one_batch(prompts):
        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=GOLD_MAX_NEW_TOKENS,
                do_sample=GOLD_TEMPERATURE > 0,
                temperature=max(GOLD_TEMPERATURE, 1e-5) if GOLD_TEMPERATURE > 0 else 1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = out[:, inputs['input_ids'].shape[1]:]
        responses = tokenizer.batch_decode(generated, skip_special_tokens=True)

        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        token_lens = (generated != pad_id).sum(dim=1).tolist()
        return responses, token_lens

    predictions = []
    bs = max(1, int(GOLD_BATCH_SIZE))
    num_votes = max(1, int(GOLD_NUM_SAMPLES))
    i = 0

    while i < len(examples):
        batch = examples[i:i + bs]
        prompts = []
        for row in batch:
            messages = [{'role': 'user', 'content': build_prompt(row['question'], row['answer'])}]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            prompts.append(text)

        try:
            t0 = time.time()

            # True parallel voting: expand prompts and generate all votes in one call.
            expanded_prompts = []
            owner_idx = []
            for local_i, prompt in enumerate(prompts):
                for _ in range(num_votes):
                    expanded_prompts.append(prompt)
                    owner_idx.append(local_i)

            expanded_responses, expanded_token_lens = generate_one_batch(expanded_prompts)
            elapsed = time.time() - t0

            votes_by_sample = [[] for _ in range(len(batch))]
            responses_by_sample = [[] for _ in range(len(batch))]
            token_lens_by_sample = [[] for _ in range(len(batch))]

            for resp, tok_len, src_i in zip(expanded_responses, expanded_token_lens, owner_idx):
                pred_lbl = parse_label(resp)
                votes_by_sample[src_i].append(pred_lbl)
                responses_by_sample[src_i].append(resp)
                token_lens_by_sample[src_i].append(int(tok_len))

            for j, row in enumerate(batch):
                vote_counter = Counter(votes_by_sample[j])
                pred = vote_counter.most_common(1)[0][0]
                true = gold_labels[row['index']]
                match = pred == true

                max_tokens_used = max(token_lens_by_sample[j]) if token_lens_by_sample[j] else 0
                hit_max_tokens = max_tokens_used >= (GOLD_MAX_NEW_TOKENS - 1)

                record = {
                    'sample_idx': row['index'],
                    'true': true,
                    'pred': pred,
                    'match': match,
                    'votes': dict(vote_counter),
                    'vote_labels': votes_by_sample[j],
                    'question': row['question'],
                    'answer': row['answer'],
                    'response': responses_by_sample[j][0] if responses_by_sample[j] else '',
                    'responses_all': responses_by_sample[j],
                    'token_lens_all': token_lens_by_sample[j],
                    'max_tokens_used': int(max_tokens_used),
                    'hit_max_tokens': bool(hit_max_tokens),
                    'batch_size_used': bs,
                    'num_samples': num_votes,
                    'batch_latency_sec': round(elapsed, 4),
                }
                predictions.append(record)
                print(f'[{row["index"]+1:03d}/{len(examples)}] True={true} | Pred={pred} | {"✅" if match else "❌"} | votes={dict(vote_counter)} | max_toks={max_tokens_used}')

            i += len(batch)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and bs > 1:
                print(f'⚠️ OOM at batch_size={bs}. Falling back to batch_size=1 and retrying...')
                bs = 1
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise

    pred_df = pd.DataFrame(predictions).sort_values('sample_idx').reset_index(drop=True)
    correct = int(pred_df['match'].sum())
    total = len(pred_df)
    acc = correct / max(1, total)

    print(f'\n🏁 GOLD EVAL Accuracy: {correct}/{total} = {acc:.1%}')

    capped_rate = float(pred_df['hit_max_tokens'].mean()) if len(pred_df) else 0.0
    p95_tokens = int(pred_df['max_tokens_used'].quantile(0.95)) if len(pred_df) else 0
    print(f'📏 Token usage: p95={p95_tokens}, cap_rate={capped_rate:.1%} (max_new_tokens={GOLD_MAX_NEW_TOKENS})')
    if capped_rate > 0.10:
        print('⚠️ More than 10% of samples hit token cap. Consider increasing GOLD_MAX_NEW_TOKENS (e.g., 384 or 512).')

    cm = pd.crosstab(pred_df['true'], pred_df['pred'], rownames=['true'], colnames=['pred'], dropna=False)
    print('\nConfusion matrix:')
    print(cm)

    out_json = OUTPUT_DIR / 'gold_eval_predictions.json'
    out_csv = OUTPUT_DIR / 'gold_eval_predictions.csv'
    out_cm = OUTPUT_DIR / 'gold_eval_confusion_matrix.csv'

    pred_df.to_json(out_json, orient='records', indent=2, force_ascii=False)
    pred_df.to_csv(out_csv, index=False)
    cm.to_csv(out_cm)

    print(f'\n💾 Saved: {out_json}')
    print(f'💾 Saved: {out_csv}')
    print(f'💾 Saved: {out_cm}')
else:
    print('⏭️ Skipping SemEval gold evaluation')

✅ Gold labels: /kaggle/input/datasets/gigibot/evalsetsemevalpolitical/task1_eval_labels.txt
✅ Eval CSV:    /kaggle/input/datasets/gigibot/claritysemevalevaldataset/clarity_task_evaluation_dataset.csv
📏 Alignment OK: 237 samples (index 0..236)
Gold label distribution: {'Direct Reply': 85, 'Direct Non-Reply': 35, 'Indirect': 117}
[001/237] True=Direct Reply | Pred=Direct Non-Reply | ❌ | votes={'Indirect': 1, 'Direct Non-Reply': 2} | max_toks=208
[002/237] True=Direct Non-Reply | Pred=Direct Non-Reply | ✅ | votes={'Direct Non-Reply': 3} | max_toks=264
[003/237] True=Indirect | Pred=Direct Non-Reply | ❌ | votes={'Direct Non-Reply': 3} | max_toks=260
[004/237] True=Direct Reply | Pred=Direct Reply | ✅ | votes={'Direct Reply': 3} | max_toks=278
[005/237] True=Indirect | Pred=Direct Reply | ❌ | votes={'Direct Reply': 2, 'Direct Non-Reply': 1} | max_toks=280
[006/237] True=Indirect | Pred=Direct Non-Reply | ❌ | votes={'Direct Non-Reply': 3} | max_toks=209
[007/237] True=Indirect | Pred=Direct 

In [8]:
### Cell 6: Train!

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=torch.cuda.is_available(),  # Use bf16 for 4-bit training
    report_to='none',
    dataloader_num_workers=0,
    optim='paged_adamw_8bit' if LOAD_4BIT else 'adamw_torch',  # Use paged optimizer only with quantization
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print('🚀 Starting training...')
if RESUME_FROM_CHECKPOINT:
    print(f'📂 Resuming from checkpoint: {RESUME_FROM_CHECKPOINT}')
    trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
else:
    trainer.train()
print('✅ Training complete!')

# Save final model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'💾 Model saved to: {OUTPUT_DIR}')

🚀 Starting training...


Step,Training Loss,Validation Loss
50,3.982320,2.800694


✅ Training complete!
💾 Model saved to: /kaggle/working/granite_lora_trained


In [ ]:
### Cell 7a: GRPO Training (optional - run AFTER SFT for reasoning improvement)
# GRPO optimizes the model to prefer better reasoning chains
# Only run this after you have a decent SFT baseline

RUN_GRPO = True  # Set True to run GRPO training

if RUN_GRPO:
    from trl import GRPOConfig, GRPOTrainer
    
    print('🧠 Starting GRPO (reasoning optimization)...')
    
    # Reward function aligned to SemEval objective:
    # predict the correct class among Direct Reply / Direct Non-Reply / Indirect.
    def reward_fn(prompts, completions, completion_ids=None, target_label=None, **kwargs):
        def extract_text(completion):
            if isinstance(completion, str):
                return completion
            if isinstance(completion, dict):
                if isinstance(completion.get('content'), str):
                    return completion['content']
                return json.dumps(completion, ensure_ascii=False)
            if isinstance(completion, list) and completion:
                last = completion[-1]
                if isinstance(last, dict) and isinstance(last.get('content'), str):
                    return last['content']
            return str(completion)

        allowed = {'Direct Reply', 'Direct Non-Reply', 'Indirect'}
        rewards = []
        target_label = target_label or []

        for i, completion in enumerate(completions):
            text = extract_text(completion)
            true_label = target_label[i] if i < len(target_label) else None

            pred_label = None
            reasoning = ''
            parsed_json = False

            # Prefer strict JSON parsing first
            try:
                data = json.loads(text)
                parsed_json = isinstance(data, dict)
                if parsed_json:
                    pred_label = data.get('label')
                    reasoning = str(data.get('reasoning', ''))
            except Exception:
                pass

            # Fallback label extraction from free text
            if pred_label not in allowed:
                lower = text.lower()
                for lbl in ['Direct Reply', 'Direct Non-Reply', 'Indirect']:
                    if lbl.lower() in lower:
                        pred_label = lbl
                        break

            score = -0.3  # base penalty for malformed outputs

            # format reward
            if parsed_json:
                score += 0.3
            if pred_label in allowed:
                score += 0.2

            # task reward (most important): correct SemEval class
            if true_label in allowed and pred_label in allowed:
                score += 0.8 if pred_label == true_label else -0.6

            # small reasoning quality bonus (secondary)
            if reasoning:
                score += min(len(reasoning) / 400.0, 0.2)

            # keep rewards bounded
            rewards.append(max(-1.0, min(1.0, score)))

        return rewards
    
    # Prepare prompts for GRPO with ground-truth labels for reward calculation
    grpo_rows = [
        {
            'prompt': build_prompt(r['question'], r['answer']),
            'target_label': r['label'],
        }
        for r in train_rows[:200]
    ]
    grpo_dataset = Dataset.from_list(grpo_rows)
    
    grpo_config = GRPOConfig(
        output_dir=str(OUTPUT_DIR / 'grpo'),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,  # Lower LR for fine-tuning
        num_generations=2,  # Generate 2 responses per prompt
        logging_steps=10,
        bf16=torch.cuda.is_available(),
        report_to='none',
    )
    
    grpo_trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        processing_class=tokenizer,
        reward_funcs=reward_fn,
        train_dataset=grpo_dataset,
    )
    
    grpo_trainer.train()
    print('✅ GRPO training complete!')
    
    # Save GRPO-enhanced model
    model.save_pretrained(OUTPUT_DIR / 'grpo')
    print(f'💾 GRPO model saved to: {OUTPUT_DIR / "grpo"}')
else:
    print('⏭️ GRPO training skipped (set RUN_GRPO=True after SFT training)')

🧠 Starting GRPO (reasoning optimization)...


Step,Training Loss
10,0.097542
20,0.000000


In [15]:
### Cell 7: Quick Evaluation
import re

model.eval()

def predict(q, a):
    prompt = build_prompt(q, a)
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Parse label from response
    try:
        data = json.loads(response)
        return data.get('label', 'Indirect')
    except:
        for lbl in ['Direct Reply', 'Direct Non-Reply', 'Indirect']:
            if lbl.lower() in response.lower():
                return lbl
        return 'Indirect'

# Evaluate on validation set
n_eval = min(20, len(val_rows))
correct = 0
print(f'Evaluating on {n_eval} examples...')

for i, row in enumerate(val_rows[:n_eval]):
    pred = predict(row['question'], row['answer'])
    true = row['label']
    match = pred == true
    correct += match
    print(f'{i+1}. True={true}, Pred={pred}, {"✅" if match else "❌"}')

print(f'\nAccuracy: {correct}/{n_eval} = {correct/n_eval:.1%}')

Evaluating on 13 examples...
1. True=Direct Non-Reply, Pred=Direct Non-Reply, ✅
2. True=Indirect, Pred=Direct Non-Reply, ❌


KeyboardInterrupt: 

In [ ]:
### Cell 8: Push to HuggingFace (optional)
from huggingface_hub import HfApi, login

HF_REPO = 'gigibot/granite-clarity-lora'  # Change this!
hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    api = HfApi()
    print(f'📤 Pushing to {HF_REPO}...')
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=HF_REPO,
        repo_type='model',
    )
    print(f'✅ Pushed to https://huggingface.co/{HF_REPO}')
else:
    print('⚠️ No HF_TOKEN found. Add to Kaggle Secrets to push model.')
    print(f'Model saved locally at: {OUTPUT_DIR}')